<a href="https://colab.research.google.com/github/natasya783/Kelola-Kendaraan/blob/main/Kelola%20Kendaraan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 2. KELOLA KENDARAAN (Bobot 15%)
# =====================================================================
def input_angka(pesan, minimum=0):
    """Meminta input bilangan bulat dan memvalidasi batas minimum."""
    while True:
        try:
            nilai = int(input(pesan))
            if nilai < minimum:
                print(f"ERROR: Nilai minimal adalah {minimum}.")
            else:
                return nilai
        except ValueError:
            print("ERROR: Masukkan angka bulat yang valid.")

def tambah_kendaraan(user_id):
    """Menyimpan profil kendaraan milik pengguna."""
    print("\n=== TAMBAH KENDARAAN ===")
    nopol = input("Nomor polisi: ").strip().upper()
    merk = input("Merk motor: ").strip()
    tipe = input("Tipe motor: ").strip()
    tahun = input_angka("Tahun kendaraan: ", minimum=1900)
    km = input_angka("Kilometer saat ini: ", minimum=0)

    if not all([nopol, merk, tipe]):
        print("ERROR: Nomor polisi, merk, dan tipe wajib diisi.")
        return

    with koneksi_db() as conn:
        conn.execute("""
            INSERT INTO kendaraan
            (user_id, nomor_polisi, merk, tipe, tahun, kilometer_saat_ini)
            VALUES (?, ?, ?, ?, ?, ?)
        """, (user_id, nopol, merk, tipe, tahun, km))
        conn.commit()
    print("Data kendaraan berhasil disimpan.")

def ambil_kendaraan(user_id):
    """Mengambil seluruh kendaraan milik pengguna yang sedang login."""
    with koneksi_db() as conn:
        cur = conn.execute("""
            SELECT id, nomor_polisi, merk, tipe, tahun, kilometer_saat_ini
            FROM kendaraan WHERE user_id = ? ORDER BY id
        """, (user_id,))
        return cur.fetchall()

def lihat_kendaraan(user_id):
    """Menampilkan daftar kendaraan pengguna."""
    data = ambil_kendaraan(user_id)
    print("\n=== DAFTAR KENDARAAN ===")
    if not data:
        print("Belum ada kendaraan. Silakan tambah kendaraan terlebih dahulu.")
        return
    print(f"{'ID':<4} {'NOMOR POLISI':<16} {'MERK':<12} {'TIPE':<18} {'TAHUN':<7} {'KM':>9}")
    for row in data:
        print(f"{row[0]:<4} {row[1]:<16} {row[2]:<12} {row[3]:<18} {row[4]:<7} {row[5]:>9,}")

def pilih_kendaraan(user_id):
    """Memilih kendaraan berdasarkan ID dan memastikan kepemilikannya."""
    data = ambil_kendaraan(user_id)
    if not data:
        print("Belum ada kendaraan. Tambahkan kendaraan terlebih dahulu.")
        return None
    lihat_kendaraan(user_id)
    id_kendaraan = input_angka("Masukkan ID kendaraan: ", minimum=1)
    for kendaraan in data:
        if kendaraan[0] == id_kendaraan:
            return kendaraan
    print("ERROR: ID kendaraan tidak ditemukan pada akun ini.")
    return None

def update_kilometer(user_id):
    """Memperbarui odometer tanpa mengizinkan kilometer berkurang."""
    print("\n=== UPDATE KILOMETER ===")
    kendaraan = pilih_kendaraan(user_id)
    if not kendaraan:
        return
    km_baru = input_angka("Kilometer baru: ", minimum=0)
    km_lama = kendaraan[5]
    if km_baru < km_lama:
        print("ERROR: Kilometer tidak boleh berkurang.")
        return
    with koneksi_db() as conn:
        conn.execute(
            "UPDATE kendaraan SET kilometer_saat_ini = ? WHERE id = ? AND user_id = ?",
            (km_baru, kendaraan[0], user_id)
        )
        conn.commit()
    print("Kilometer berhasil diperbarui.")